# Three business questions on the gold layer (epic 5)

Answers three business questions from the gold tables (RF-05,
[ADR 012](../docs/adr/012-gold-aggregations-and-counting-rules.md)): openings per year, UF and
CNAE; active companies by size and region; sectors with the most closed or unfit companies.

**Where the data and the outputs come from.** The lake lives in MinIO inside a GitHub runner
(ADR 006; no real data on the development laptop). The manual
[`Gold analysis`](../.github/workflows/gold-analysis.yml) workflow downloads a release subset,
runs `make bronze`, `make silver` and `make gold` (sum checks against silver),
exports small roll-ups of the three gold tables to `data/*.csv`
(`scripts/gold_export.py`) and executes this notebook on them with `nbconvert`. The executed
notebook and the CSVs are the workflow artifact and are committed as is. They hold aggregates
only (counts by year, UF, CNAE, size, status x sector): no rows, names or CNPJs (RST-01/LGPD).
The first cell prints the reference month, the files loaded and the silver counts.

**Scope: a sample, not the full release.** The default subset is `lookups` + `Empresas1` +
`Estabelecimentos1` of 2026-09. `Estabelecimentos1` (~350 MB zipped) is about **6.5% of the
establishment files by size** (5.38 GB in ten files, of which `Estabelecimentos0` alone is
2.24 GB, [data source](../docs/data-source.md)), and `Empresas1` about 6% of the companies.
**Absolute numbers are therefore partial; only shares, rates and rankings are meaningful**, and
only as far as the files are a fair sample of the release (the source does not document how
records are split across files). Companies are joined to establishments by `cnpj_basico`
within the loaded files only: `silver.without_company` in the first cell counts the loaded
establishments whose company is in another `Empresas` file (they are left out of the size
table, ADR 012). To widen the sample, run the workflow with more files (`files` input); the
runner has ~14 GB of disk.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.1f}".format
DATA = Path("data")
KEYS = ["uf", "cnae_fiscal_principal", "porte_empresa", "situacao_cadastral", "cnae_divisao"]
dtype = dict.fromkeys(KEYS, str)


def load(name: str) -> pd.DataFrame:
    return pd.read_csv(DATA / f"{name}.csv", dtype=dtype)


meta = json.loads((DATA / "export.json").read_text(encoding="utf-8"))
ano_uf, cnae, porte, uf, setor = (
    load(n) for n in ("q1_ano_uf", "q1_cnae", "q2_porte", "q2_uf", "q3_situacao_setor")
)
r0, r1 = meta["recent_years"]
print(json.dumps(meta, indent=1))

## 1. How many companies were opened per year, UF and CNAE?

Source: `gold_empresas_cnae_uf_ano` (establishments by main CNAE x UF x opening year, ADR 012).
The unit is the **establishment** (head office or branch), the one that carries the opening
date, the state and the CNAE.

In [ ]:
anos = ano_uf.dropna(subset=["ano_inicio_atividade"]).astype({"ano_inicio_atividade": int})
por_ano = anos.groupby("ano_inicio_atividade")["estabelecimentos"].sum().loc[1990:]
recentes = anos[anos["ano_inicio_atividade"].between(r0, r1)]
uf_pct = recentes.groupby("uf")["estabelecimentos"].sum()
uf_pct = (uf_pct / uf_pct.sum() * 100).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
por_ano.plot(ax=ax1, color="#2a6f97")
ax1.set(title="Establishments opened per year (since 1990)", xlabel="Opening year")
ax1.set_ylabel("Establishments (loaded files)")
uf_pct.tail(10).plot.barh(ax=ax2, color="#2a6f97")
ax2.set(title=f"Share of the {r0}-{r1} openings by UF (top 10)", xlabel="%", ylabel="")
fig.tight_layout()
plt.show()

col = f"estabelecimentos_{r0}_{r1}"
top_cnae = cnae.head(10).assign(share_pct=cnae[col] / recentes["estabelecimentos"].sum() * 100)
top_cnae[["cnae_fiscal_principal", "cnae_descricao", col, "share_pct"]]

**Conclusion.** PENDING: written from the outputs of the first `Gold analysis` run.

## 2. How are active companies distributed by size (porte) and region?

Sources: `gold_distribuicao_porte` (companies by `porte_empresa`; `empresas_ativas` = companies
with at least one active establishment in the loaded files) and `gold_empresas_cnae_uf_ano`
rolled up to UF and then to the five IBGE regions (active establishments: a company with
branches in two regions counts in both, so regions are measured in establishments).

In [ ]:
REGIAO = {
    "Norte": "AC AM AP PA RO RR TO",
    "Nordeste": "AL BA CE MA PB PE PI RN SE",
    "Centro-Oeste": "DF GO MS MT",
    "Sudeste": "ES MG RJ SP",
    "Sul": "PR RS SC",
}
uf_regiao = {u: r for r, ufs in REGIAO.items() for u in ufs.split()}
regiao = (
    uf.assign(regiao=uf["uf"].map(uf_regiao).fillna("Abroad / not informed"))
    .groupby("regiao")[["estabelecimentos", "estabelecimentos_ativos"]]
    .sum()
    .assign(
        share_of_active_pct=lambda d: d.estabelecimentos_ativos
        / d.estabelecimentos_ativos.sum()
        * 100,
        active_rate_pct=lambda d: d.estabelecimentos_ativos / d.estabelecimentos * 100,
    )
    .sort_values("estabelecimentos_ativos")
)
porte_ativo = porte.assign(
    porte=porte["porte_descricao"].fillna("NULL / unknown"),
    share_of_active_pct=porte.empresas_ativas / porte.empresas_ativas.sum() * 100,
    active_rate_pct=porte.empresas_ativas / porte.empresas * 100,
).set_index("porte")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
porte_ativo["share_of_active_pct"].sort_values().plot.barh(ax=ax1, color="#2a6f97")
ax1.set(title="Active companies by size (porte), %", xlabel="%", ylabel="")
regiao["share_of_active_pct"].plot.barh(ax=ax2, color="#2a6f97")
ax2.set(title="Active establishments by region, %", xlabel="%", ylabel="")
fig.tight_layout()
plt.show()

display(porte_ativo[["empresas", "empresas_ativas", "share_of_active_pct", "active_rate_pct"]])
regiao.iloc[::-1]

**Conclusion.** PENDING: written from the outputs of the first `Gold analysis` run.

## 3. Which sectors have the most closed (baixadas) or unfit (inaptas) companies?

Source: `gold_situacao_setor` (establishments by `situacao_cadastral` x CNAE section/division of
the main CNAE). "Closed or unfit" = status `08` BAIXADA + `04` INAPTA. Two views: the volume
(which sectors concentrate them) and the rate within the sector (how many of the sector's
establishments are closed or unfit).

In [ ]:
setor = setor.fillna(
    {"cnae_secao": "-", "cnae_secao_descricao": "CNAE not informed", "situacao_descricao": "OTHER"}
)
by = setor.pivot_table(
    index=["cnae_secao", "cnae_secao_descricao"],
    columns="situacao_descricao",
    values="estabelecimentos",
    aggfunc="sum",
    fill_value=0,
)
by = by.reindex(columns=sorted(set(by.columns) | {"ATIVA", "BAIXADA", "INAPTA"}), fill_value=0)
by["total"] = by.sum(axis=1)
by["baixadas_inaptas"] = by["BAIXADA"] + by["INAPTA"]
by["share_of_closed_pct"] = by.baixadas_inaptas / by.baixadas_inaptas.sum() * 100
by["closed_rate_pct"] = by.baixadas_inaptas / by.total * 100
by = by.sort_values("baixadas_inaptas", ascending=False)
top = by.head(10).iloc[::-1]
labels = [f"{s} {d[:38]}" for s, d in top.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
ax1.barh(labels, top["BAIXADA"], color="#2a6f97", label="BAIXADA (08)")
ax1.barh(labels, top["INAPTA"], left=top["BAIXADA"], color="#e07a5f", label="INAPTA (04)")
ax1.set(title="Closed or unfit establishments (top 10 sections)", xlabel="Establishments")
ax1.legend(loc="lower right")
ax2.barh(labels, top["closed_rate_pct"], color="#81b29a")
ax2.set(title="Closed or unfit as % of the section", xlabel="%")
fig.tight_layout()
plt.show()

cols = ["total", "BAIXADA", "INAPTA", "share_of_closed_pct", "closed_rate_pct"]
by[cols].head(10).reset_index()

**Conclusion.** PENDING: written from the outputs of the first `Gold analysis` run.